# Section 3 : Classification Supervisée

Dans cette section, nous appliquons quatre algorithmes de classification :
1. **$k$-Nearest Neighbors ($k$-NN)** 
2. **Multinomial Naïve Bayes** 
3. **
4. **

L'objectif est de prédire la recommandation d'achat des clients (`Recommended IND`) à partir des caractéristiques extraites et prétraitées des avis textuels.



### 3.1. Chargement des Données et Environnement

Nous importons les matrices creuses prétraitées ($X_{train}$, $X_{test}$) ainsi que les étiquettes cibles ($y_{train}$, $y_{test}$).

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed"


X_train = sp.load_npz(DATA_PATH / "X_train_processed.npz")
X_test = sp.load_npz(DATA_PATH / "X_test_processed.npz")
y_train = pd.read_csv(DATA_PATH / "y_train.csv").values.ravel()
y_test = pd.read_csv(DATA_PATH / "y_test.csv").values.ravel()

results = []

print("✓ Setup completed and data loaded!")

✓ Setup completed and data loaded!


### 3.2. Algorithme des $k$-Plus Proches Voisins ($k$-NN)

L'algorithme k-NN est une méthode non paramétrique basée sur la mesure de distance entre les instances. 

**Hyperparamètres optimisés via `GridSearchCV` :**
* `n_neighbors` (k) : Nombre de voisins à prendre en compte ($k \in \{5, 11, 21\}$).
* `weights` : Pondération des voisins (`uniform` ou `distance`).
* `algorithm` : Recherche par force brute (`brute`), adaptée aux matrices creuses TF-IDF.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score


PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed"

X_train = sp.load_npz(DATA_PATH / "X_train_processed.npz")
X_test = sp.load_npz(DATA_PATH / "X_test_processed.npz")
y_train = pd.read_csv(DATA_PATH / "y_train.csv").values.ravel()
y_test = pd.read_csv(DATA_PATH / "y_test.csv").values.ravel()


results = []


param_grid_knn = {
    "n_neighbors": [5, 11, 21],
    "weights": ["uniform", "distance"],
    "algorithm": ["brute"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid_knn, 
    cv=3, 
    scoring='f1', 
    n_jobs=2
)


grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

try:
    auc_knn = roc_auc_score(y_test, best_knn.predict_proba(X_test)[:, 1])
except Exception:
    auc_knn = None

results.append({
    "Model": "K-Nearest Neighbors",
    "Best Params": grid_knn.best_params_,
    "Accuracy": round(acc_knn, 4),
    "F1-Score": round(f1_knn, 4),
    "ROC-AUC": round(auc_knn, 4) if auc_knn else "N/A"
})

print("=== K-Nearest Neighbors (k-NN) ===")
print(f"Best Hyperparameters: {grid_knn.best_params_}")
print(f"Accuracy: {acc_knn:.4f} | F1-Score: {f1_knn:.4f} | ROC-AUC: {auc_knn if auc_knn else 'N/A'}\n")
print(classification_report(y_test, y_pred_knn))

=== K-Nearest Neighbors (k-NN) ===
Best Hyperparameters: {'algorithm': 'brute', 'n_neighbors': 21, 'weights': 'uniform'}
Accuracy: 0.9186 | F1-Score: 0.9519 | ROC-AUC: 0.9642336920823539

              precision    recall  f1-score   support

           0       0.87      0.64      0.74       834
           1       0.93      0.98      0.95      3859

    accuracy                           0.92      4693
   macro avg       0.90      0.81      0.84      4693
weighted avg       0.92      0.92      0.91      4693



### 3.3. Chapitre 5 : Classifieur Naïve Bayes (Multinomial)

Le classifieur Naïve Bayes repose sur le théorème de Bayes avec une hypothèse forte d'indépendance conditionnelle entre les variables. Le modèle **Multinomial Naïve Bayes** est particulièrement adapté aux fréquences de mots et représentations TF-IDF.

**Traitement spécifique & Hyperparamètres :**
* Tronquage des valeurs négatives (`np.clip`) pour garantir la compatibilité avec la distribution multinomiale.
* Optimisation du paramètre de lissage de Laplace ($\alpha \in \{0.01, 0.1, 0.5, 1.0, 2.0\}$).

In [ ]:
from sklearn.naive_bayes import MultinomialNB


X_tr_nb = X_train.copy()
X_tr_nb.data = np.clip(X_tr_nb.data, 0, None)

X_te_nb = X_test.copy()
X_te_nb.data = np.clip(X_te_nb.data, 0, None)


param_grid_nb = {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0]}

grid_nb = GridSearchCV(MultinomialNB(), param_grid_nb, cv=5, scoring='f1', n_jobs=-1)
grid_nb.fit(X_tr_nb, y_train)


best_nb = grid_nb.best_estimator_
y_pred_nb = best_nb.predict(X_te_nb)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, best_nb.predict_proba(X_te_nb)[:, 1])

results.append({
    "Model": "Multinomial Naive Bayes",
    "Best Params": grid_nb.best_params_,
    "Accuracy": round(acc_nb, 4),
    "F1-Score": round(f1_nb, 4),
    "ROC-AUC": round(auc_nb, 4)
})

print("=== Multinomial Naïve Bayes ===")
print(f"Best Hyperparameters: {grid_nb.best_params_}")
print(f"Accuracy: {acc_nb:.4f} | F1-Score: {f1_nb:.4f} | ROC-AUC: {auc_nb:.4f}\n")
print(classification_report(y_test, y_pred_nb))

=== Multinomial Naïve Bayes ===
Best Hyperparameters: {'alpha': 0.1}
Accuracy: 0.9052 | F1-Score: 0.9425 | ROC-AUC: 0.9531

              precision    recall  f1-score   support

           0       0.74      0.72      0.73       834
           1       0.94      0.95      0.94      3859

    accuracy                           0.91      4693
   macro avg       0.84      0.83      0.84      4693
weighted avg       0.90      0.91      0.90      4693



les autres models (hadil)

### 3.4. Synthèse et Comparaison des Modèles

Le tableau ci-dessous récapitule les meilleures configurations d'hyperparamètres ainsi que les performances obtenues sur l'ensemble de test en termes d'**Accuracy**, **F1-Score** et **ROC-AUC**.

In [ ]:

comparison_df = pd.DataFrame(results)
display(comparison_df)

,Model,Best Params,Accuracy,F1-Score,ROC-AUC
0,K-Nearest Neighbors,"{'algorithm': 'brute', 'n_neighbors': 21, 'wei...",0.9186,0.9519,0.9642
1,Multinomial Naive Bayes,{'alpha': 0.1},0.9052,0.9425,0.9531
